In [4]:
# import sys
# print(sys.path)

['/home/icb/francesca.drummer/1-Projects/GT-long-range-niches/docs/notebooks', '/home/icb/francesca.drummer/miniconda3/envs/exphormer_env/lib/python39.zip', '/home/icb/francesca.drummer/miniconda3/envs/exphormer_env/lib/python3.9', '/home/icb/francesca.drummer/miniconda3/envs/exphormer_env/lib/python3.9/lib-dynload', '', '/home/icb/francesca.drummer/miniconda3/envs/exphormer_env/lib/python3.9/site-packages', '/home/icb/francesca.drummer/1-Projects/GT-long-range-niches/src']


In [1]:
from graph_transformer_long_range_niches.tl.utils import pad_batch
from graph_transformer_long_range_niches.model.gnn_transformer import GNNTransformer
from graph_transformer_long_range_niches.tl.test_datasets import CustomGraphDataset_nodeLabel, CustomGraphDataset_graphLabel

import yaml
import torch
from torch_geometric.loader import DataLoader

/home/icb/francesca.drummer/miniconda3/envs/exphormer_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Create dataset + loader
num_graphs = 10
max_nodes_per_graph = 50
max_edges_per_graph = 30
nr_node_features = 20
BATCH_SIZE = 5
cfg_path = '/home/icb/francesca.drummer/1-Projects/GT-long-range-niches/src/config_files/test.yaml'

with open(cfg_path) as stream:
    cfg_dict = yaml.safe_load(stream)

nodeLabel_custom_dataset = CustomGraphDataset_nodeLabel(num_graphs, max_nodes_per_graph, max_edges_per_graph, nr_node_features, cfg_dict["dataset"]["num_classes"])

for i in range(num_graphs):
    data = nodeLabel_custom_dataset[i]
    print(f"Graph {i + 1} - Nodes: {data.num_nodes}, Edges: {data.num_edges}, Label: {data.y}")

train_loader = DataLoader(nodeLabel_custom_dataset.data_list, batch_size=BATCH_SIZE) # merges batch_size graphs from a PyG dataset 

Graph 1 - Nodes: 26, Edges: 25, Label: tensor([3, 3, 2, 2, 3, 0, 1, 0, 0, 3, 1, 1, 1, 3, 1, 3, 1, 0, 1, 1, 1, 2, 0, 3,
        2, 0])
Graph 2 - Nodes: 32, Edges: 25, Label: tensor([0, 3, 3, 3, 0, 2, 2, 2, 1, 0, 0, 3, 3, 3, 2, 2, 0, 1, 0, 3, 0, 2, 1, 2,
        1, 0, 3, 3, 1, 0, 3, 1])
Graph 3 - Nodes: 2, Edges: 0, Label: tensor([0, 3])
Graph 4 - Nodes: 9, Edges: 23, Label: tensor([0, 2, 0, 1, 1, 0, 2, 1, 3])
Graph 5 - Nodes: 15, Edges: 25, Label: tensor([0, 2, 0, 1, 3, 3, 2, 0, 3, 2, 1, 1, 2, 0, 2])
Graph 6 - Nodes: 21, Edges: 1, Label: tensor([0, 2, 3, 1, 2, 0, 0, 2, 1, 0, 0, 3, 3, 2, 1, 3, 1, 1, 0, 2, 2])
Graph 7 - Nodes: 20, Edges: 9, Label: tensor([1, 1, 1, 0, 1, 0, 2, 0, 1, 1, 0, 3, 0, 3, 3, 1, 2, 0, 0, 0])
Graph 8 - Nodes: 42, Edges: 19, Label: tensor([1, 0, 1, 1, 0, 0, 0, 3, 2, 2, 3, 2, 3, 1, 3, 1, 0, 3, 2, 3, 0, 1, 0, 0,
        2, 1, 2, 1, 2, 3, 2, 3, 2, 0, 0, 3, 0, 2, 1, 1, 2, 2])
Graph 9 - Nodes: 36, Edges: 0, Label: tensor([1, 3, 1, 3, 2, 3, 1, 0, 3, 1, 3, 1, 0, 0, 0, 1, 2,

In [3]:
# Calculate accuracy
def accuracy(pred_y, y):
    return (pred_y == y).sum() / len(y)

def train(train_loader, model, optimizer, criterion, N_EPOCHS=20):
    # Data for animations
    embeddings = []
    losses = []
    accuracies = []
    outputs = []

    # Training loop
    for epoch in range(N_EPOCHS):
        for idx, batch in enumerate(train_loader): 
            # Clear gradients
            optimizer.zero_grad()

            # Forward pass
            out = model(batch) # [B, C] with C being the number of tasks to predict, e.i. 
            print('Out put GNN+Transformer: ', out.shape, '/n Batch shape: ', batch.y.shape)

            # Calculate loss function
            loss = criterion(out, batch.y)

            # Calculate accuracy
            print('Predicted label: ', out.argmax(dim=1), 'True label', batch.y)
            acc = accuracy(out.argmax(dim=1), batch.y)

            # Compute gradients
            loss.backward(retain_graph=True)

            # Tune parameters
            optimizer.step()

            # Store data for animations
            #embeddings.append(h)
            losses.append(loss)
            accuracies.append(acc)
            outputs.append(out.argmax(dim=1))

            # Print metrics every 10 epochs
            if epoch % 10 == 0:
                print(f'Epoch {epoch:>3} | Loss: {loss:.2f} | Acc: {acc*100:.2f}%')

In [4]:
cfg_path = '/home/icb/francesca.drummer/1-Projects/GT-long-range-niches/src/config_files/test.yaml'

with open(cfg_path) as stream:
    cfg_dict = yaml.safe_load(stream)
    
gnn_transformer_model = GNNTransformer(cfg_dict)
lr = 0.1
weight_decay = 5e-4
optimizer = torch.optim.Adam(gnn_transformer_model.parameters(), lr=lr, weight_decay=weight_decay)
criterion = torch.nn.CrossEntropyLoss()

In [5]:
train(train_loader, gnn_transformer_model, optimizer, criterion, N_EPOCHS=20)

GNN out:  torch.Size([84, 8]) z torch.Size([84, 4])
GNN predicted node label accuracy:  tensor(0.3095)
After gnn2transformer:  torch.Size([84, 128])
After Pad:  torch.Size([32, 5, 128])
Input padded_h_node:  torch.Size([32, 5, 128])
CLS + Normalized padded_h_node:  torch.Size([33, 5, 128])
Padding mask:  torch.Size([5, 33])
TransformerEncoder output:  torch.Size([33, 5, 128])
hgraph output:  torch.Size([32, 5, 128])
Permuted h_graph: torch.Size([5, 32, 128])
Padding mask: torch.Size([5, 32])
masked output:  tensor([[[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
           0.0000e+00, -0.0000e+00],
         [ 0.0000e+00,  0.0000e+00, -0.0000e+00,  ..., -0.0000e+00,
           0.0000e+00,  0.0000e+00],
         [ 0.0000e+00,  0.0000e+00, -0.0000e+00,  ...,  0.0000e+00,
           0.0000e+00,  0.0000e+00],
         ...,
         [-6.6982e-01,  1.4910e+00, -2.4625e-01,  ...,  8.4335e-01,
          -9.7321e-02, -9.7458e-02],
         [-1.2684e-01,  1.3738e+00,  5.0298e-01,  ...

TypeError: sum() received an invalid combination of arguments - got (out=NoneType, axis=int, ), but expected one of:
 * (*, torch.dtype dtype)
      didn't match because some of the keywords were incorrect: out, axis
 * (tuple of ints dim, bool keepdim, *, torch.dtype dtype)
 * (tuple of names dim, bool keepdim, *, torch.dtype dtype)


## Dataset

In [4]:
import torch
from torch import nn, Tensor
from torch_geometric.data import Data, Dataset
import random

In [11]:
class CustomGraphDataset_graphLabel(Dataset):
    def __init__(self, num_graphs, max_nodes, max_edges, nr_node_features, nr_classes):
        self.num_graphs = num_graphs
        self.max_nodes = max_nodes
        self.max_edges = max_edges
        self.nr_node_features = nr_node_features
        self.data_list = []

        for _ in range(num_graphs):
            num_nodes = random.randint(1, max_nodes)
            num_edges = random.randint(0, min(num_nodes * (num_nodes - 1) // 2, max_edges))
            
            edge_index = torch.zeros((2, num_edges), dtype=torch.long)
            for i in range(num_edges):
                edge_index[0, i] = random.randint(0, num_nodes - 1)
                edge_index[1, i] = random.randint(0, num_nodes - 1)
            
            x = torch.rand((num_nodes, nr_node_features), dtype=torch.float)  # Node features (random for example)
            y = torch.tensor([random.randint(0, nr_classes)], dtype=torch.long)  # Graph label (binary for example)

            data = Data(x=x, edge_index=edge_index, y=y)
            self.data_list.append(data)

    def __len__(self):
        return self.num_graphs

    def __getitem__(self, idx):
        return self.data_list[idx]

In [7]:
class CustomGraphDataset_nodeLabel(Dataset):
    def __init__(self, num_graphs, max_nodes, max_edges, nr_node_features, nr_classes):
        self.num_graphs = num_graphs
        self.max_nodes = max_nodes
        self.max_edges = max_edges
        self.nr_node_features = nr_node_features
        self.nr_classes = nr_classes
        self.data_list = []

        for _ in range(num_graphs):
            num_nodes = random.randint(1, max_nodes)
            num_edges = random.randint(0, min(num_nodes * (num_nodes - 1) // 2, max_edges))
            
            edge_index = torch.zeros((2, num_edges), dtype=torch.long)
            for i in range(num_edges):
                edge_index[0, i] = random.randint(0, num_nodes - 1)
                edge_index[1, i] = random.randint(0, num_nodes - 1)
            
            x = torch.rand((num_nodes, nr_node_features), dtype=torch.float)  # Node features (random for example)
            y = torch.randint(0, nr_classes, (num_nodes,), dtype=torch.long)  # Node labels

            data = Data(x=x, edge_index=edge_index, y=y)
            self.data_list.append(data)

    def __len__(self):
        return self.num_graphs

    def __getitem__(self, idx):
        return self.data_list[idx]

Graph 0:
Node features: tensor([[0.6526, 0.0120, 0.9608, 0.9571],
        [0.1992, 0.2248, 0.6153, 0.8605]])
Edge connections: tensor([], size=(2, 0), dtype=torch.int64)
Node labels: tensor([1, 2])

Graph 1:
Node features: tensor([[0.6774, 0.6035, 0.9178, 0.3928]])
Edge connections: tensor([], size=(2, 0), dtype=torch.int64)
Node labels: tensor([1])

Graph 2:
Node features: tensor([[0.9264, 0.5861, 0.3614, 0.0699],
        [0.4331, 0.5820, 0.6580, 0.8271]])
Edge connections: tensor([], size=(2, 0), dtype=torch.int64)
Node labels: tensor([2, 1])



In [12]:
num_graphs = 10
max_nodes_per_graph = 50
max_edges_per_graph = 30
nr_node_features = 20

graphLabel_custom_dataset = CustomGraphDataset_graphLabel(num_graphs, max_nodes_per_graph, max_edges_per_graph, nr_node_features, cfg_dict["dataset"]["num_classes"])

for i in range(num_graphs):
    data = graphLabel_custom_dataset[i]
    print(f"Graph {i + 1} - Nodes: {data.num_nodes}, Edges: {data.num_edges}, Label: {data.y.item()}")

Graph 1 - Nodes: 23, Edges: 19, Label: 3
Graph 2 - Nodes: 40, Edges: 26, Label: 4
Graph 3 - Nodes: 37, Edges: 21, Label: 3
Graph 4 - Nodes: 25, Edges: 4, Label: 3
Graph 5 - Nodes: 9, Edges: 16, Label: 3
Graph 6 - Nodes: 30, Edges: 5, Label: 0
Graph 7 - Nodes: 40, Edges: 23, Label: 1
Graph 8 - Nodes: 1, Edges: 0, Label: 1
Graph 9 - Nodes: 42, Edges: 10, Label: 0
Graph 10 - Nodes: 7, Edges: 7, Label: 4


In [31]:
nodeLabel_custom_dataset = CustomGraphDataset_nodeLabel(num_graphs, max_nodes_per_graph, max_edges_per_graph, nr_node_features, cfg_dict["dataset"]["num_classes"])

for i in range(num_graphs):
    data = nodeLabel_custom_dataset[i]
    print(f"Graph {i + 1} - Nodes: {data.num_nodes}, Edges: {data.num_edges}, Label: {data.y}")

Graph 1 - Nodes: 27, Edges: 2, Label: tensor([3, 0, 3, 2, 3, 3, 0, 0, 1, 0, 2, 1, 1, 1, 1, 0, 1, 1, 2, 3, 2, 2, 3, 1,
        0, 1, 2])
Graph 2 - Nodes: 50, Edges: 0, Label: tensor([2, 1, 1, 1, 1, 2, 2, 1, 1, 1, 3, 2, 3, 0, 2, 0, 2, 3, 1, 2, 1, 2, 3, 2,
        3, 0, 3, 3, 2, 0, 0, 1, 1, 0, 3, 3, 2, 0, 0, 3, 0, 1, 3, 0, 1, 3, 1, 0,
        2, 0])
Graph 3 - Nodes: 2, Edges: 1, Label: tensor([3, 3])
Graph 4 - Nodes: 21, Edges: 3, Label: tensor([3, 1, 0, 0, 1, 3, 0, 3, 0, 2, 1, 1, 3, 2, 0, 2, 3, 3, 2, 2, 1])
Graph 5 - Nodes: 27, Edges: 28, Label: tensor([1, 1, 2, 0, 3, 0, 1, 3, 3, 3, 1, 1, 3, 2, 1, 2, 2, 1, 0, 3, 0, 2, 0, 0,
        3, 1, 1])
Graph 6 - Nodes: 8, Edges: 18, Label: tensor([3, 2, 0, 3, 1, 3, 1, 3])
Graph 7 - Nodes: 1, Edges: 0, Label: tensor([2])
Graph 8 - Nodes: 48, Edges: 2, Label: tensor([0, 3, 2, 2, 3, 2, 3, 3, 0, 1, 2, 1, 0, 3, 3, 3, 2, 1, 0, 3, 3, 2, 1, 1,
        3, 1, 2, 2, 3, 1, 1, 2, 3, 0, 1, 3, 0, 3, 3, 0, 2, 2, 1, 0, 2, 3, 1, 3])
Graph 9 - Nodes: 36, Edges: 0, La

## Dataloader 

In [32]:
from torch_geometric.loader import DataLoader

In [33]:
BATCH_SIZE = 5
train_loader = DataLoader(nodeLabel_custom_dataset.data_list, batch_size=BATCH_SIZE) # merges batch_size graphs from a PyG dataset 

In [34]:
for idx, batch in enumerate(train_loader):
    print(batch)

DataBatch(x=[127, 20], edge_index=[2, 34], y=[127], batch=[127], ptr=[6])
DataBatch(x=[99, 20], edge_index=[2, 23], y=[99], batch=[99], ptr=[6])


## Config file

In [3]:
import yaml

In [3]:
cfg_path = '/home/icb/francesca.drummer/1-Projects/GT-long-range-niches/src/config_files/test.yaml'

In [4]:
with open(cfg_path) as stream:
    cfg_dict = yaml.safe_load(stream)

In [5]:
cfg_dict

{'out_dir': 'results',
 'dataloader': {'data_path': '', 'radius': 30},
 'dataset': {'prediction_task': 'node', 'num_classes': 4},
 'gnn': {'gnn_type': 'GCN',
  'num_layers': 2,
  'num_features': 20,
  'hidden_dim': 16,
  'embed_dim': 8},
 'transformer': {'d_model': 128,
  'n_heads': 4,
  'dim_feedforward': 512,
  'dropout': 0.3,
  'num_layers': 4,
  'activation_func': 'relu',
  'num_encoder_layers': 4}}

## GCN

In [35]:
from torch.nn import Linear
from torch_geometric.nn import GCNConv, MessagePassing
import torch.nn.functional as F

class GCN(torch.nn.Module):
    def __init__(self, 
                 cfg,
                 num_classes,
                 dp_rate = 0.1):
        super().__init__()      
        #dp_rate = cfg['dp_rate'] if cfg['dp_rate'] is not None else dp_rate
        self.num_classes = num_classes

        in_dim, hidden_dim = cfg['num_features'], cfg['hidden_dim']
        layers = []
        for l_idx in range(cfg['num_layers'] - 1):
            layers += [
                GCNConv(in_channels=in_dim, out_channels=hidden_dim),
                nn.ReLU(inplace=True),
                nn.Dropout(dp_rate)
            ]
            in_dim = hidden_dim
        
        layers += [GCNConv(in_channels=in_dim, out_channels=cfg['embed_dim'])]
        self.layers = nn.ModuleList(layers)
        self.out = Linear(cfg['embed_dim'], num_classes)

    def forward(self, x, edge_index):
        """
        Input:
            x: Adjacency matrix (n x obs)
            edge_index: gene expressiong (var x obs)
        """
        for layer in self.layers:
            if isinstance(layer, MessagePassing):
                x = layer(x, edge_index)
            else:
                x = layer(x)
        h = F.relu(x)
        z = self.out(h)
        return x, z


In [36]:
model_GCN = GCN(cfg_dict['gnn'], cfg_dict["dataset"]["num_classes"])
N_EPOCHS = 20
lr = 0.1
weight_decay = 5e-4
optimizer = torch.optim.Adam(model_GCN.parameters(), lr=lr, weight_decay=weight_decay)

In [37]:
model_GCN

GCN(
  (layers): ModuleList(
    (0): GCNConv(20, 16)
    (1): ReLU(inplace=True)
    (2): Dropout(p=0.1, inplace=False)
    (3): GCNConv(16, 8)
  )
  (out): Linear(in_features=8, out_features=4, bias=True)
)

In [41]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_GCN.parameters(), lr=0.02)

# Calculate accuracy
def accuracy(pred_y, y):
    return (pred_y == y).sum() / len(y)

# Data for animations
embeddings = []
losses = []
accuracies = []
outputs = []

# Training loop
for epoch in range(N_EPOCHS):
    for idx, batch in enumerate(train_loader): 
        # Clear gradients
        optimizer.zero_grad()

        # Forward pass
        h, z = model_GCN(batch.x, batch.edge_index)
        # print('Embedding GNN: ', h.shape)
        # print('Prediction GNN: ', z.shape)

        # Calculate loss function
        loss = criterion(z, batch.y)

        # Calculate accuracy
        acc = accuracy(z.argmax(dim=1), batch.y)

        # Compute gradients
        loss.backward()

        # Tune parameters
        optimizer.step()

        # Store data for animations
        embeddings.append(h)
        losses.append(loss)
        accuracies.append(acc)
        outputs.append(z.argmax(dim=1))

        # Print metrics every 10 epochs
        if epoch % 10 == 0:
            print(f'Epoch {epoch:>3} | Loss: {loss:.2f} | Acc: {acc*100:.2f}%')
            print(h.shape, z.shape)

Epoch   0 | Loss: 1.06 | Acc: 56.69%
torch.Size([127, 8]) torch.Size([127, 4])
Epoch   0 | Loss: 1.06 | Acc: 55.56%
torch.Size([99, 8]) torch.Size([99, 4])
Epoch  10 | Loss: 0.97 | Acc: 62.20%
torch.Size([127, 8]) torch.Size([127, 4])
Epoch  10 | Loss: 0.91 | Acc: 67.68%
torch.Size([99, 8]) torch.Size([99, 4])


In [25]:
embeddings[-2].shape

torch.Size([140, 8])

## Transformer 

In [26]:
class TransformerNodeEncoder(nn.Module):
    """
    Sequence of: Dropout → Layer Norm → FC → nonlinearity → Dropout → FC → Dropout → Layer Norm + residual connections
    """
    
    def __init__(self, cfg):

        super().__init__()

        # Save model parameters
        self.model_type = 'TransformerEncoder'
        self.max_input_len = 1000
        self.d_model = cfg['d_model']
        self.n_heads = cfg['n_heads']
        self.dropout = cfg['dropout']
        self.act_func = cfg['activation_func']
        self.num_encoded_layers = cfg['num_encoder_layers']
        self.dim_feedforward = cfg['dim_feedforward']

        ## ToDo print model parameters

        # Create Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            self.d_model, self.n_heads, self.dim_feedforward, self.dropout, self.act_func
        )
        encoder_norm = nn.LayerNorm(self.d_model)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, self.num_encoded_layers, norm=encoder_norm)

        self.norm_input = nn.LayerNorm(self.d_model)
        self.cls_embedding = nn.Parameter(torch.randn([1, 1, self.d_model], requires_grad=True))
        

    def forward(self, padded_h_node, src_padding_mask):
        """
        Input: 
            padded_h_node: [n_b x B X h_d] with n_b: dimension of batch, B: batch size, h_d: dimension of transformer
            padding_mask: [B x n_b] matrix indicating the size of the padding mask to be ignored during calculation 
        """
        print('Input padded_h_node: ', padded_h_node.shape)
        
        # append cls embedding
        expand_cls_embedding = self.cls_embedding.expand(1, padded_h_node.size(1), -1)
        padded_h_node = torch.cat([padded_h_node, expand_cls_embedding], dim=0)
        # normalize input
        padded_h_node = self.norm_input(padded_h_node)

        print('CLS + Normalized padded_h_node: ', padded_h_node.shape)

        zeros = src_padding_mask.data.new(src_padding_mask.size(0), 1).fill_(0)
        src_padding_mask = torch.cat([src_padding_mask, zeros], dim=1)

        print("Padding mask: ", src_padding_mask.shape)

        transformer_out = self.transformer_encoder(padded_h_node, src_key_padding_mask=src_padding_mask)  # (S, B, h_d)
        return transformer_out, src_padding_mask

In [27]:
import torch
import torch.nn as nn

# Sample input dimensions
batch_size = 16
seq_len = 20
embedding_dim = 64

# Sample input
padded_h_node = torch.randn(seq_len, batch_size, embedding_dim)
src_padding_mask = torch.zeros(batch_size, seq_len, dtype=torch.bool)
src_padding_mask[1, 10:] = True  # Example: mask positions from index 10 to the end for batch 1

# Model configuration
model_config = {
    'd_model': embedding_dim,
    'n_heads': 4,
    'dropout': 0.1,
    'activation_func': 'relu',
    'num_encoder_layers': 6,
    'dim_feedforward': 256
}

# Instantiate the model
transformer_node_encoder = TransformerNodeEncoder(model_config)

# Forward pass
transformer_out, new_src_padding_mask = transformer_node_encoder(padded_h_node, src_padding_mask)

# Check output dimensions
print("Output shape:", transformer_out.shape)
print("New source padding mask shape:", new_src_padding_mask.shape)

Input padded_h_node:  torch.Size([20, 16, 64])
CLS + Normalized padded_h_node:  torch.Size([21, 16, 64])
Padding mask:  torch.Size([16, 21])
Output shape: torch.Size([21, 16, 64])
New source padding mask shape: torch.Size([16, 21])


# GNN + Transformer

In [31]:
class GNNTransformer(nn.Module):
    """
    Sequence of: Dropout → Layer Norm → FC → nonlinearity → Dropout → FC → Dropout → Layer Norm + residual connections
    """
    
    def __init__(self, cfg):

        super().__init__()

        self.model_type = 'GNN_Transformer'
        self.output_dim = cfg["transformer"]["d_model"]

        self.gnn2transformer = nn.Linear(cfg['gnn']['embed_dim'], cfg['transformer']['d_model'])
        self.norm_input = nn.LayerNorm(cfg['transformer']['d_model'])
        self.cls_embedding = nn.Parameter(torch.randn([1, 1, cfg['transformer']['d_model']], requires_grad = True))
        self.num_tasks = cfg['gnn']['num_classes']
        self.max_seq_len = None
        
        # GNN initialization
        self.gnn = GCN(cfg["gnn"])
        # Transformer encoder initialization
        self.transformer_encoder = TransformerNodeEncoder(cfg["transformer"])

        self.graph_pred_linear_list = torch.nn.ModuleList()
        if self.max_seq_len is None:
            self.graph_pred_linear = torch.nn.Linear(self.output_dim, self.num_tasks)
        else:
            for i in range(self.max_seq_len):
                self.graph_pred_linear_list.append(torch.nn.Linear(self.output_dim, self.num_tasks))
        

    def forward(self, batched_data):
        """
        Input: 
            src: final per-node GNN encodings ``[seq_len/batch x embed_dim_gnn]``
            src_mask: final per-node GNN encodings ``[var/genes x var/genes]``
        """
        h_node, _ = self.gnn(batched_data.x, batched_data.edge_index)
        print('GNN out: ',h_node.shape)
        h_node = self.gnn2transformer(h_node)  # [s, b, d_model]
        print('After gnn2transformer: ', h_node.shape)

        padded_h_node, src_padding_mask, num_nodes, mask, max_num_nodes = pad_batch(
            h_node, batched_data.batch, self.transformer_encoder.max_input_len, get_mask=True
        )  # Pad in the front

        print("After Pad: ", padded_h_node.shape)

        transformer_out = padded_h_node
        transformer_out, _ = self.transformer_encoder(transformer_out, src_padding_mask)  # [s, B, h], [B, s]
        print('TransformerEncoder output: ', transformer_out.shape)
        # get cls 
        h_graph = transformer_out[-1]

        if self.max_seq_len is None:
            out = self.graph_pred_linear(h_graph)
            return out
        
        pred_list = []
        for i in range(self.max_seq_len):
            pred_list.append(self.graph_pred_linear_list[i](h_graph))

        return pred_list

In [32]:
gnn_transformer_model = GNNTransformer(cfg_dict)
N_EPOCHS = 20
lr = 0.1
weight_decay = 5e-4
optimizer = torch.optim.Adam(gnn_transformer_model.parameters(), lr=lr, weight_decay=weight_decay)
criterion = torch.nn.CrossEntropyLoss()

In [35]:
# Calculate accuracy
def accuracy(pred_y, y):
    return (pred_y == y).sum() / len(y)

# Data for animations
embeddings = []
losses = []
accuracies = []
outputs = []

# Training loop
for epoch in range(N_EPOCHS):
    for idx, batch in enumerate(train_loader): 
        # Clear gradients
        optimizer.zero_grad()

        # Forward pass
        out = gnn_transformer_model(batch) # [B, C] with C being the number of tasks to predict, e.i. 

        print('Out put GNN+Transformer: ', out.shape, '/n Batch shape: ', batch.y.shape)

        # Calculate loss function
        loss = criterion(out, batch.y)

        # Calculate accuracy
        acc = accuracy(z.argmax(dim=1), data.y)

        # Compute gradients
        loss.backward(retain_graph=True)

        # Tune parameters
        optimizer.step()

        # Store data for animations
        embeddings.append(h)
        losses.append(loss)
        accuracies.append(acc)
        outputs.append(z.argmax(dim=1))

        # Print metrics every 10 epochs
        if epoch % 10 == 0:
            print(f'Epoch {epoch:>3} | Loss: {loss:.2f} | Acc: {acc*100:.2f}%')

GNN out:  torch.Size([140, 8])
After gnn2transformer:  torch.Size([140, 128])
After Pad:  torch.Size([40, 5, 128])
Input padded_h_node:  torch.Size([40, 5, 128])
CLS + Normalized padded_h_node:  torch.Size([41, 5, 128])
Padding mask:  torch.Size([5, 41])
TransformerEncoder output:  torch.Size([41, 5, 128])
Out put GNN+Transformer:  torch.Size([5, 4]) /n Batch shape:  torch.Size([5])
Epoch   0 | Loss: 1.75 | Acc: 100.00%
GNN out:  torch.Size([52, 8])
After gnn2transformer:  torch.Size([52, 128])
After Pad:  torch.Size([25, 5, 128])
Input padded_h_node:  torch.Size([25, 5, 128])
CLS + Normalized padded_h_node:  torch.Size([26, 5, 128])
Padding mask:  torch.Size([5, 26])
TransformerEncoder output:  torch.Size([26, 5, 128])
Out put GNN+Transformer:  torch.Size([5, 4]) /n Batch shape:  torch.Size([5])
Epoch   0 | Loss: 1.28 | Acc: 100.00%
GNN out:  torch.Size([140, 8])
After gnn2transformer:  torch.Size([140, 128])
After Pad:  torch.Size([40, 5, 128])
Input padded_h_node:  torch.Size([40, 5